In [1]:
import Scope

Eventually, you'll want to run a computational workflow for a given system.  

To do so, you need: 

1) To **configure scope** in a computation cluster (i.e. a cluster with a job manager, SLURM or SGE). You can also configure scope in your local computer, to analyze the data, but you won't be able to submit computations. 
 
    To do it, you can run the script "**scope_config**" in either/both the local computer and the computation cluster, and follow instructions.

    ```bash
    scope_config -h
    ```

    This will create a environment-class object that will contain all the relevant information for the job submission. 

2) To **create a system** and add sources (molecules, or unit cells).  

3) To **create a job file**, as a plain text file, for easier interaction. 

4) Then, you can run SCOPE in a computation cluster (i.e. a cluster with a job manager, SLURM or SGE), with one -or many- job files, and a given system. To do so, you can use the "scope_run_job" script in the computation cluster: 

    ```bash
    scope_run_job -h
    ```

# Example:

## 1) Create a Dummy Environment

In [ ]:
## !!!!!!!!
## Here, we create a dummy environment, with the minimal information to complete this tutorial
## Normally, you should configure the environment correctly in your local computer or in a computation cluster
## To run the job at the end of this tutorial, an environment configured in a computation cluster, with Gaussian 16, will be needed
## !!!!!!!!

import os
env = Scope.environment(name="tutorial4")
env.sources_path  = ''.join([os.path.abspath('.'),"/Data","/Tutorial_4","/Sources/"])
env.sys_path      = ''.join([os.path.abspath('.'),"/Data","/Tutorial_4","/Systems/"])
env.calcs_path    = ''.join([os.path.abspath('.'),"/Data","/Tutorial_4","/Calcs/"])

#os.makedirs(env.sources_path, exist_ok=True)
#os.makedirs(env.sys_path, exist_ok=True)
#os.makedirs(env.calcs_path, exist_ok=True)

ENVIRONMENT.SET_MANAGEMENT_TYPE: Could not recognise the Queue Management System
ENVIRONMENT.SET_MANAGEMENT_TYPE: Assuming this is a local computer
ENVIRONMENT.SET_MANAGEMENT_TYPE: If this is a computation cluster with SLURM or SGE, please report bug


In [ ]:
# Optional, you can save the environment anywhere. To keep folders tidy, I'll save it inside the Tutorial_4 folder
env.save(''.join([os.path.abspath('.'),"/Data","/Tutorial_4/scope_env_tutorial4.npy"]))

## 2.1) Create a System

In [4]:
## Now we create a system for a water molecule
from Scope.Classes_System import system
test_sys = system("water", env)
print(test_sys)

---------------------------------
   >>> SCOPE System Object >>>   
---------------------------------
 Version               = 1.0
 Type                  = system
 Subtype               = system
 Name                  = water
 Source Path           = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_4/Sources/water/
 Calculations Path     = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_4/Calcs/water/
 System File Path      = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_4/Systems/water/




## 2.2) Create a Molecule and Associate it to the system

In [5]:
## Now we create an actual water molecule, whose geometry is read from an .xyz file
from Scope.Classes_Specie import molecule
from Scope.Read_Write import read_xyz

labels, coords = read_xyz(test_sys.sources_path+"water.xyz")
molec = molecule(labels, coords)
print(molec)

--------------------------------------------------
------------- SCOPE MOLECULE Object --------------
--------------------------------------------------
 Version               = 1.0
 Type                  = specie
 Sub-Type              = molecule
 Number of Atoms       = 3
 Formula               = H-O2
 Number of Parents     = 0
 Total Charge          = None
 Has Adjacency Matrix  = NO 
 Has Bonds             = NO




In [6]:
## Create The Initial State, from which we will draw the data for the computations.
ini_state = molec.add_state("initial")
ini_state.set_geometry(labels, coords)

In [7]:
## And source it to our system, giving it the name "reference"
_ = test_sys.add_source('reference',molec)
## Notice that the molecule has been added:
print(test_sys)

---------------------------------
   >>> SCOPE System Object >>>   
---------------------------------
 Version               = 1.0
 Type                  = system
 Subtype               = system
 Name                  = water
 Source Path           = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_4/Sources/water/
 Calculations Path     = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_4/Calcs/water/
 System File Path      = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_4/Systems/water/

 # of Sources          = 1
     idx, type, name, formula               
     0: specie reference H-O2 




In [8]:
## Trying to add new molecules with the same name won't work, to avoid duplicities
_ = test_sys.add_source('reference',molec, debug=1)

FIND_SOURCE. Searching for source with reference in system with 1 sources:
    reference specie
ADD_SOURCE: Source with name 'reference' already exists in system
If you would like to Overwrite, specify overwrite=True


## 2.3) Save the system

In [9]:
## Finally, save the system to a file:
test_sys.save()

## 3) Create a Job File

In [10]:
from Scope.Classes_Input import *

In [11]:
## Job Specs can be created from a file, or from a string. 
## This is an example input read as a string. We will soon go through the different parts
## First, notice that there are four sections, initiated with '&', and terminated with '/'.
## Sections can be empty as in the example below (see &options) or even absent. In those cases, default values will be added
## Also, notice that the values are either introduced as strings 'X', lists [], booleans, or integers/floats...
## Please respect the notation used for each type of variable

test_input = """
&environment
   max_jobs         = 80
   max_procs        = 240
   requested_procs  = 8
/

&options
/

&job_data
   branch       = 'Isolated'
   recipe       = ['reference']
   setup        = 'regular'
   suffix       = 'opt'
   keyword      = 'pbe opt'

   istate       = 'initial'
   fstate       = 'pbe opt'

   hierarchy    = 1
   requisites   = []
   constrains   = ['self']
   must_be_good = False
/

&qc_data
   software     = 'g16'
   jobtype      = 'opt'
   functional   = 'pbe'
   basis        = 'def2SVP'
   is_grimme    = True
   loose_opt    = True
/"""

In [12]:
# IMPORTANT. Notice that, when running the 'scope_run_job' script. The job file (-j arg) must be an actual file, not a string as in the cell above
# Thus, we save this text to an actual file, so we can call it later
from Scope.Read_Write import save_text
file_name = ''.join([os.path.abspath('.'),"/Data","/Tutorial_4/test_job.job"])
save_text(test_input, file_name)

In [13]:
## In any case, the job can be interpreted and stored as INPUT_DATA class objects:
## There are 4 input sections. Which are read separately 
local_env   = input_data(content=test_input, section="&environment", isfile=False)
options     = input_data(content=test_input, section="&options", isfile=False)
job_data    = input_data(content=test_input, section="&job_data", isfile=False)
qc_data     = input_data(content=test_input, section="&qc_data", isfile=False)

## They could be read in a single line (see below), but the data would not be stored correctly 
## inp = input_data(content=test_input, isfile=False)  

INPUT_DATA.READ: Section '&options' is empty


### 3.1) Local Environment Options

In [14]:
# Here are the options related to the computation resources, that are not hardcoded during the environment creation. 
# The user can change these options anytime in the job file, without having to modify the environment
local_env

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.max_jobs            | <class 'int'>       | 80        
self.max_procs           | <class 'int'>       | 240       
self.requested_procs     | <class 'int'>       | 8         

In [15]:
# There are some defaults as well, which are always added to the user options
local_env = fill_environment_data(local_env)
print(local_env)

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.max_jobs            | <class 'int'>       | 80        
self.max_procs           | <class 'int'>       | 240       
self.requested_procs     | <class 'int'>       | 8         
self.method              | <class 'str'>       | weighted  



In [16]:
## Notice that a "method" variable has been added. This one specifies how best queue is chosen if more than one is available

### 3.2) Options

In [17]:
## These are other options related to the submission:
options = fill_options_data(options)
print(options)

## This section was empty in the "test_input", but now defaults were added:

# want_submit:  
    # Here, the user can turn off submission (want_submit = False) of jobs. 
    # If so, input files will be created, but computations won't be submitted. 
    # This is perfect for testing the input files creation

# ignore_submitted: 
    # Normally when submitting a computation, SCOPE verifies that no computation with the same name is already running. 
    # If activated (ignore_submitted = True), this check won't be done, and multiple instances of the same job will be created.
    # This might be OK in some cases, when those job, even if they have the same name, refer to different input files.

# overwrite_inputs and overwrite_logs:
    # When true, the code can overwrite files that already exist.

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.want_submit         | <class 'bool'>      | True      
self.ignore_submitted    | <class 'bool'>      | False     
self.overwrite_inputs    | <class 'bool'>      | True      
self.overwrite_outputs   | <class 'bool'>      | True      



### 3.3) Job Data

In [18]:
## This section includes job information. 
# 
## Basically, a Job with this information will be created (if it doesn't exist yet) or found (if it already exists) inside each system's workflow
## and other related information: 
job_data = fill_job_data(job_data)
print(job_data)

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.branch              | <class 'str'>       | Isolated  
self.recipe              | <class 'list'>      | ['reference']
self.setup               | <class 'str'>       | regular   
self.suffix              | <class 'str'>       | opt       
self.keyword             | <class 'str'>       | pbe_opt   
self.istate              | <class 'str'>       | initial   
self.fstate              | <class 'str'>       | pbe_opt   
self.hierarchy           | <class 'int'>       | 1         
self.requisites          | <class 'list'>      | []        
self.constrains          | <class 'list'>      | ['self']  
self.must_be_good        | <class 'bool'>      | False     



In [19]:
## Here's the meaning of each variable:

# branch:       to locate the job inside the system, the BRANCH must be specified. 
# recipe:       to locate the job inside the system, the RECIPE must be specified. If more than one, one job will be created inside each recipe 
# setup:        how are COMPUTATIONS inside the job created
# suffix:       which string must be added to the input/output/sub files to identify. For example, in ABITEM_opt_r1_HS.log, "opt" is the suffix
# keyword:      a name given to this job. 
# istate:       state from which the information must be extracted 
# fstate:       state where the information will be stored
# hierarchy:    expected order of jobs inside recipe 
# requisites:   list of keywords of jobs that should have finished before this one can start
# constrains:   list of keywords of jobs that should have NOT finished before this one can start
# must_be_good: whether the job can considered done even if it did not finish correctly. 
    # if must_be_good = true, it will try hard to do the task. For instance, if the user requested an optimization, it will not stop until reaching the minimum
    # if must_be_good = false, it will perform just one attempt

### 3.4) QC data

In [20]:
## Finally, we have the information of the actual Quantum Chemistry (QC) computation.
qc_data = fill_qc_data(qc_data)
print(qc_data)

## This is the information that will be passed to the QC software to create the input 

## In principle, you can update the information here (eg. you can change the basis set) and re-submit. 
## If you do so, and everything works well, the code will compare the old and new qc_datas and if an update is detected...
## ...it will resubmit even if the same job under the old qc_data was flagged as complete.
## In other words, it should be save to update the qc_data, but I haven't tested all changes.

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.software            | <class 'str'>       | g16       
self.jobtype             | <class 'str'>       | opt       
self.functional          | <class 'str'>       | pbe       
self.basis               | <class 'str'>       | def2svp   
self.is_grimme           | <class 'bool'>      | True      
self.loose_opt           | <class 'bool'>      | True      
self.tight_opt           | <class 'bool'>      | False     
self.grimme_type         | <class 'str'>       | d2        



In [21]:
## Currently, the code accepts a very limited set of qc_data options. Only Quantum Espresso and Gaussian codes are implemented, 
## together with a very limited set of functionals, basis sets, etc...

In [22]:
## In Quantum Espresso, we have implemented different Pseudo-Potential Libraries available in the literature. 
## These are called in this section ('pseudo' variable). 
## See below for an example of the qc_data for Quantum Espresso:

#&qc_data
#   software     = 'qe'
#   version      = '7.0'
#   jobtype      = 'vc-relax'
#   functional   = 'pbe'
#   is_hubbard   = True
#   is_grimme    = True
#   grimme_type  = 'd3bj'
#   uterm        = 2.35
#   mix_beta     = 0.7
#   elec_conv    = 1e-5
#   pseudo       = 'vanderbilt'
#/

## 4) Run

In [23]:
## Once the environment, system and job_file are prepared, we can submit the computation in the appropriate cluster
## To do so, we can use the 'scope_run_job' script

Once the environment, system and job_file are prepared, we can submit the job in the appropriate computation cluster.

This computation cluster should have **Gaussian 16** installed, and a **configured environment**. 
You can give it any name, but for the sake of simplicity, here we will call it **"scope_env_cluster.npy"**  

Then, we can use the 'scope_run_job' script. If you didn't change the SYSTEM and JOB_FILE paths used in this tutorial, it should be like that:

1) Go to the Tutorial_4 folder inside a computation cluster, and:
2) ```bash
   scope_run_job -n scope_env_cluster.npy -s Systems/water/water.npy -j test_job.job
   ```

In [27]:
## The environment of the computation cluster (scope_env_cluster.npy) will automatically update the paths stored in the system.
## It will use the function:
test_sys.read_paths_from_environment?

Signature: test_sys.read_paths_from_environment(environment: object, debug: int = 0) -> None
Docstring:
Modifies the paths associated with all branches, recipes, jobs, and computation files of a system
Based on the paths stored in the environment object
Args:
    environment (object): The environment to which the system paths must be updated
    debug (int, optional): Debug level. Defaults to 0.
Returns:
    bool: Returns True if the system paths were updated. Otherwise false
File:      ~/Documents/SCOPE/Program/Scope_New/src/Scope/Classes_System.py
Type:      method